# Generating Feature and Label Tiles from Classified LiDAR and Vector Data

This notebook demonstrates how to process a large collection of classified LiDAR files (.las / .laz) into individual feature tiles derived from LiDAR data, and how to generate corresponding raster label tiles from vector data. 

Additional preprocessing steps such as High Pass Median Filtering (HPMF), rasterization with vector buffering, and patch (chip) generation are applied to enhance terrain features and create standardized datasets for machine learning workflows.

## Workflow:

---

1. **File search**
   
   Recursively search for all `.las` and `.laz` files in the working directory or sub-folders.
   
---

2. **DEM parameters**
   
   Define interpolation settings such as resolution, output type (IDW), search radius, power parameter, and window size.
   
---

3. **Per-file processing**
   
   For each LiDAR file:
   - Read only ground-classified points (classification code = 2).  
   - Apply statistical outlier removal to clean the point cloud.  
   - Interpolate to raster with Inverse Distance Weighting (IDW), creating one Digital Elevation Model (DEM) tile per input file.

    <br>

    The tile-based approach is well-suited for handling very large LiDAR datasets, since each file is processed independently without overloading RAM.
     
---

4. **High Pass Median Filter (HPMF)**
   
   Apply a high-pass median filter to the DEM tiles to highlight local elevation changes.  
   The method subtracts each cell value from the median of its neighbourhood, emphasizing fine-scale terrain variability while suppressing broad trends.
   
---

5. **Rasterization**
   
   Each DEM tile is used as a mask to extract the matching area from a large vector dataset and rasterize it into a label tile.  
   Before rasterization, vector geometries are buffered (1.5 m) to ensure coverage of narrow or thin features.  
   The buffered geometries are then rasterized onto the DEM grid.  

   To avoid creating unrealistic fixed-width labels, the rasterized geometries are combined with the HPMF output:  
   only pixels within the buffered vector lines **and** with an HPMF value below –0.075 are kept as ditch pixels.
   This ensures that labels follow real terrain depressions rather than forming uniform strips around the vector lines **(threshold can be adjusted)**.

   Finally, a majority filter is applied to remove isolated spurious pixels and smooth the label shapes.  
   
---

6.  **Chip generation**
   
    Each HPMF and label raster is first resampled from 2000 × 2000 to 2048 × 2048 pixels to ensure dimensions divisible by the desired chip size.
    The resampled rasters are then divided into smaller, non-overlapping 512 × 512 px chips to create standardized training samples.
    Each HPMF chip is min–max normalized to the range [0, 1], while label chips retain their binary 0/1 values.

    Temporary intermediate files are deleted after processing, and the resulting chips are saved into dedicated subdirectories for downstream model training.


---

7. **Output**
   
    Results are written to dedicated subdirectories:

    - `model_input_data/dem_tiles` → DEM rasters generated from IDW interpolation  
    - `model_input_data/hpmf_chips` → Normalized HPMF chips for ML training  
    - `model_input_data/label_chips` → Corresponding label chips aligned with HPMF tiles

    <br>
      
    These standardized and spatially consistent chips are directly compatible with the **U-Net** convolutional neural network architecture used by **Lidberg et al. (2023)** for semantic segmentation of drainage ditches from LiDAR-derived HPMF data.


## Environment Setup and Imports

On Windows I recommend to use a dedicated **Conda environment** for this workflow, because installing **PDAL** and its dependencies can be problematic on Windows.
With Conda, installation is much simpler since most geospatial libraries (PDAL, GDAL, etc.) are available via the `conda-forge` channel.

---

Create and activate the environment:

```bash
conda create -n ditchnet_preprocessing -c conda-forge python=3.11 pdal numpy geopandas shapely rasterio opencv tifffile whitebox scikit-learn jupyter

conda activate ditchnet_preprocessing
```

Alternatively, you can use the preconfigured environment file included in this repository:

```bash
conda env create -f ditchnet_preprocessing.yml

conda activate ditchnet_preprocessing

```
---

Once the environment is active, you can import the necessary Python libraries in the notebook:

In [1]:
import pdal
from pathlib import Path
import json
import numpy as np
import shutil

import geopandas as gpd
from shapely.geometry import box
import rasterio
from rasterio import features

import cv2
import tifffile as tiff

from whitebox.whitebox_tools import WhiteboxTools
from sklearn.preprocessing import MinMaxScaler

wbt = WhiteboxTools()
wbt.verbose = False

### LiDAR Preprocessing and DEM Generation

File search

In [2]:
# Current working directory absolute path
data_dir = Path().resolve()
# Recursive search of *.laz or *.las files in all sub-folders
las_files = list(data_dir.rglob("*.laz")) + list(data_dir.rglob("*.las"))
las_files = [str(f) for f in las_files]
print(f"Found {len(las_files)} files")

Found 10 files


Define DEM parameters

In [3]:
# DEM parameters
resolution = 0.5
output_type = "min" # or "idw", but "min" probably better for our purpose
radius = 1.0 # 1 m, we can try different values
power = 2.0 # only for idw
window_size = 5 # if there are no points in radius, how many surrounding values use for interpolation

Set output directiories

In [4]:
dem_dir = data_dir / "dem_tiles"

temp_dir = data_dir / "model_input_data" / "temp"
temp_hpmf_dir = temp_dir / "hpmf_tiles"
temp_label_dir = temp_dir / "label_tiles"

hpmf_chip_dir = data_dir / "model_input_data" / "hpmf_chips"
label_chip_dir = data_dir / "model_input_data" / "label_chips"

# Create data folders if they don't exist
dem_dir.mkdir(parents=True, exist_ok=True)

temp_hpmf_dir.mkdir(parents=True, exist_ok=True)
temp_label_dir.mkdir(parents=True, exist_ok=True)

hpmf_chip_dir.mkdir(parents=True, exist_ok=True)
label_chip_dir.mkdir(parents=True, exist_ok=True)

Process each file using PDAL pipeline and save as DEM tiles

In [5]:
# For each file (enumerate just so we can track how many files has been processed)
for i, las in enumerate(las_files, 1):
    las_path = Path(las)
    # name of DEM file
    dem_file = dem_dir / f"{las_path.stem}_dem_{output_type}.tif"
    # Tracking progress
    print(f"[{i}/{len(las_files)}] Processing {las_path.name} → {dem_file.name}")

    # input for PDAL is a JSON which can be done as a dictionary in python and then converting to JSON
    pipeline_dict = {
        "pipeline": [{"type": "readers.las", "filename": str(las_path)}, # read file     
            {"type": "filters.range", "limits": "Classification[2:2]"},  # only ground class
            {"type": "filters.outlier", "method": "statistical", "mean_k": 8, "multiplier": 2.5}, # filter outlying points
            {
                "type": "writers.gdal",  # create DEM with chosen parameters
                "filename": str(dem_file),
                "resolution": resolution,
                "output_type": output_type,
                "radius": radius,
                "power": power,
                "window_size": window_size,
                "gdaldriver": "GTiff"
            }
        ]
    }
    # convert dictionary to JSON and run pipeline
    pipeline = pdal.Pipeline(json.dumps(pipeline_dict))
    count = pipeline.execute()

print(f"Pipeline finished. Created {len(las_files)} DEM files")

[1/10] Processing P4433B2_3.laz → P4433B2_3_dem_min.tif
[2/10] Processing P4433B2_4.laz → P4433B2_4_dem_min.tif
[3/10] Processing P4433B2_6.laz → P4433B2_6_dem_min.tif
[4/10] Processing P4433B2_7.laz → P4433B2_7_dem_min.tif
[5/10] Processing P4433B3_1.laz → P4433B3_1_dem_min.tif
[6/10] Processing P4433B3_2.laz → P4433B3_2_dem_min.tif
[7/10] Processing P4433B3_4.laz → P4433B3_4_dem_min.tif
[8/10] Processing P4433B3_7.laz → P4433B3_7_dem_min.tif
[9/10] Processing P4433B3_9.laz → P4433B3_9_dem_min.tif
[10/10] Processing P4433B4_9.laz → P4433B4_9_dem_min.tif
Pipeline finished. Created 10 DEM files


### Feature Enhancement and Label Generation (HPMF & Rasterization)

In [6]:
def minmax_normalized_image(image):
    # Handle uniform images: if all values are equal, return a zero array to avoid division by zero
    if np.max(image) == np.min(image):
        return np.zeros(image.shape, dtype=np.float32)

    # Replace no data values with ones
    image = np.where(np.isnan(image) | (image == -9999), 1, image)

    scaler = MinMaxScaler()                                               # Initialize MinMaxScaler to scale pixel values between 0 and 1
    flat_normalized_image = scaler.fit_transform(image.reshape(-1, 1))    # Flatten the image for scaler input and apply normalization
    normalized_image = flat_normalized_image.reshape(image.shape)         # Reshape the normalized data back to the original image dimensions

    return normalized_image.astype(np.float32)

In [7]:
# Load vector data (ditch lines) from GeoPackage
# !!! IMPORTANT: Replace with the path to your own vector dataset !!!
label_vector_gdf = gpd.read_file("./label_vector_data/Hytky_iisalmi.gpkg")

C:\Users\OWNER\miniconda3\envs\ditchnet_preprocessing\Lib\site-packages\pyogrio\core.py:35: RuntimeWarning: Could not detect GDAL data files. Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()


In [8]:
# Initialize index counter for naming the generated chip files
chip_idx = 0

# Iterate through all DEM tiles
for dem in dem_dir.iterdir():
    
    # Apply High Pass Median Filter (HPMF) to DEM
    hpmf_file = temp_hpmf_dir / f"{dem.stem}_hpmf.tif"
    wbt.high_pass_median_filter(i=dem, output=hpmf_file, filterx=11, filtery=11)
    
    # Open the HPMF raster and read array + metadata
    with rasterio.open(hpmf_file) as hpmf_raster:
        hpmf_array = hpmf_raster.read(1)       # Read raster values as array
        hpmf_bounds = hpmf_raster.bounds       # Get raster bounding box
        hpmf_shape = hpmf_raster.shape         # Get raster dimensions (rows, cols)
        transform = hpmf_raster.transform      # Get affine transform (pixel -> coords)

    # Create a polygon covering the HPMF tile extent
    hpmf_geom = box(hpmf_bounds.left, hpmf_bounds.bottom, hpmf_bounds.right, hpmf_bounds.top)
    hpmf_gdf = gpd.GeoDataFrame(geometry=[hpmf_geom], crs=label_vector_gdf.crs)

    # Clip vector data (ditches) to HPMF tile extent
    clipped_label_vector_gdf = gpd.clip(gdf=label_vector_gdf, mask=hpmf_gdf)

    # Buffer vector geometries (1.5 m) to give them width
    buffered_label_geom = clipped_label_vector_gdf.buffer(distance=1.5)

    # Rasterize buffered geometries onto HPMF tile grid
    buffered_label_array = features.rasterize(shapes=[(geom, 1) for geom in buffered_label_geom.geometry], # Geometries to rasterize (value=1 inside buffer)
                                              out_shape=hpmf_shape,                                        # Match output size to HPMF raster
                                              transform=transform,                                         # Align to same grid/coordinates as HPMF
                                              fill=0,                                                      # Background pixels get value 0
                                              dtype=np.uint8,                                              # Use 8-bit integer values
                                              all_touched=True)                                            # Mark all pixels touched by geometry, not just centers)

    # Combine buffered vector raster with HPMF mask
    # Keep only pixels within buffer where HPMF < -0.075 (threshold 0.00 might work better for our data)
    final_label_array = np.where((buffered_label_array == 1) & (hpmf_array < -0.075), 1, 0)

    # Save the binary label raster (0 = background, 1 = ditch)
    label_file = temp_label_dir / f"{dem.stem}_label.tif"
    tiff.imwrite(label_file, final_label_array.astype(np.uint8))

    # Apply majority filter to clean noise and smooth labels
    wbt.majority_filter(i=label_file, output=label_file, filterx=3, filtery=3)

    # Resample label raster to 2048×2048 using nearest-neighbor interpolation to preserve class values
    label_array = tiff.imread(label_file)
    label_array = cv2.resize(label_array, (2048, 2048), interpolation=cv2.INTER_NEAREST) 

    # Normalize HPMF pixel values to 0–1 range for consistent scaling
    hpmf_array = minmax_normalized_image(hpmf_array)
    
    # Resample HPMF raster to 2048×2048 using bilinear interpolation for smoother elevation transitions
    hpmf_array = cv2.resize(hpmf_array, (2048, 2048), interpolation=cv2.INTER_LINEAR)

    # ----- GENERATE NORMALIZED AND PADDED 512 × 512 RASTER CHIPS FROM TEMPORARY HPMF AND LABEL DATA -----
    
    # Define original raster size and chip size
    original_raster_size = 2048
    chip_size = 512
    
    # Loop over the raster in steps of chip_size to create non-overlapping chips
    for i in range(0, original_raster_size, chip_size):
        for j in range(0, original_raster_size, chip_size):
            
            # --- HPFM CHIPS ---
            hpmf_chip = hpmf_array[i:i + chip_size, j:j + chip_size]     # Extract a chip from the HPMF array
            
            hpmf_chip_file = hpmf_chip_dir / f"{chip_idx}.tif"           # Save the label chip as 32-bit float TIFF image
            tiff.imwrite(hpmf_chip_file, hpmf_chip.astype(np.float32))   

            # --- LABEL CHIPS ---
            label_chip = label_array[i:i + chip_size, j:j + chip_size]   # Extract a chip from the label array

            label_chip_file = label_chip_dir / f"{chip_idx}.tif"         # Save the label chip as 8-bit binary TIFF image
            tiff.imwrite(label_chip_file, label_chip.astype(np.uint8))   

            chip_idx += 1     # Increment chip index for the next output file

    hpmf_file.unlink()        # Delete temporary HPMF raster after chip generation
    label_file.unlink()       # Delete temporary label raster after chip generation

shutil.rmtree(Path(temp_dir)) # Remove the temporary working directory


### References

Lidberg, W., Paul, S. S., Westphal, F., Richter, K. F., Lavesson, N., Melniks, R., Ivanovs, J., Ciesielski, M., Leinonen, A., & Ågren, A. M. (2023).  
*Mapping Drainage Ditches in Forested Landscapes Using Deep Learning and Aerial Laser Scanning.*  
**Journal of Irrigation and Drainage Engineering**, 149(3).  
https://doi.org/10.1061/JIDEDH.IRENG-9796
